In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

#### Based on RAG Pipeline
#### Step 1. Load the document

In [ ]:
loader=PyPDFLoader("D:\\bloodtestreport.pdf")
docs=loader.load()

3

#### Step 2. Split the document

In [12]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
splitted_Docs=splitter.split_documents(docs)

#### Step 3. Do the vector embeddings

In [14]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

#### Step 4. Store the vectors in Vector DB

In [15]:
vector_store=InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=splitted_Docs
)

#### Now we will create agent part
#### For creating agent we need below things:
1. Tools
2. LLm
3. Prompt

In [44]:
# Now create the tool first
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of PDF documents, these 
        PDF documents have details about medical reports.
    """
    print("Tool called",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for data in docs:
        context += data.page_content + "\n\n"

    return context

In [45]:
llm=ChatOpenAI(model="gpt-5")

In [37]:
system_prompt="""
You are a helpful assistant that answers the questions using retrieved context.
ALWAYS use the 'retriever_tool' tool for questions requiring external knowledge.
"""

In [46]:
# Now create the agent
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [47]:
query="what is patient name, and what is the doctor name"
response=agent.invoke({"messages":[{"role":"user","content":query}]})

Tool called patient name doctor name
Tool calledTool called Reporting Doctor name
 Name: patient Name


In [43]:
print(response["messages"][-1].content)

- Patient name: Mr. Jaikey Chauhan
- Doctor name: Not specified in the report (the “Reporting Doctor” field is blank; it only notes “Referred by: Rajdhani Hospital”)
